# Fine-Tuning LLMs

Notebook for **Chapter 10** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 Fine-Tune, Prompt or RAG? The Decision Framework

## 10.2 LoRA and QLoRA: The Math

In [ ]:
import numpy as np
W=np.arange(16.).reshape(4,4)
r,alpha=1,2
B=np.zeros((4,r)); A=np.array([[0.,1,0,0]])
x=np.array([1.,2,3,4])
print("start: B=0, output equals base:",np.allclose(W@x+(alpha/r)*(B@(A@x)),W@x))
B=np.array([[1.],[0],[0],[0]])   # after some training, B changed
delta=(alpha/r)*(B@A)
print("delta W:\n",delta)
print("base out  ",W@x)
print("lora out  ",W@x+(alpha/r)*(B@(A@x)))
print("same as merged W+delta:",np.allclose((W+delta)@x, W@x+(alpha/r)*(B@(A@x))))
print("trainable numbers: A",A.size,"+ B",B.size,"=",A.size+B.size,"vs full",W.size)

start: B=0, output equals base: True
delta W:
 [[0. 2. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
base out   [ 20.  60. 100. 140.]
lora out   [ 24.  60. 100. 140.]
same as merged W+delta: True
trainable numbers: A 4 + B 4 = 8 vs full 16


## 10.3 GPU Memory Estimates With Real Numbers

In [ ]:
GB=1e9
def lora_params(layers,d_in,d_out,r,mats): return layers*mats*r*(d_in+d_out)
def mem(P,mode,trainable=0):
    if mode=="full":  b=P*(2+2+4+8)
    elif mode=="lora": b=P*2+trainable*(2+2+4+8)
    elif mode=="qlora": b=P*0.5+trainable*(2+2+4+8)
    return b/GB
P=7e9
for r in (8,16,64):
    t=lora_params(32,4096,4096,r,4)
    print(f"r={r:2d} q,k,v,o trainable={t:,} ({t/P:.2%} of 7B)")
t=lora_params(32,4096,4096,16,4)
print("full FT        ",round(mem(P,'full'),1),"GB (before activations)")
print("LoRA bf16 base ",round(mem(P,'lora',t),1),"GB")
print("QLoRA 4-bit    ",round(mem(P,'qlora',t),1),"GB")
for name,P2 in [("1B",1e9),("13B",13e9),("70B",70e9)]:
    print(name,"full",round(P2*16/GB),"GB | bf16 weights",round(P2*2/GB),"GB | 4-bit weights",round(P2*0.5/GB,1),"GB")

## 10.4 Data Preparation and Alignment

In [ ]:
import json, hashlib, re
rows=[{"instruction":"Summarise: Server crashed at 2am.","output":"The server crashed at 2am."},
{"instruction":"Summarise:  server crashed at 2am. ","output":"The server crashed at 2am."},
{"instruction":"Summarise: p95 rose to 3s.","output":""},
{"instruction":"Summarise: Disk is 90% full.","output":"The disk is nearly full."}]
norm=lambda s:re.sub(r"\s+"," ",s.strip().lower())
seen=set(); keep=[]
for r in rows:
    if not r["output"].strip(): continue
    h=hashlib.md5(norm(r["instruction"]).encode()).hexdigest()
    if h in seen: continue
    seen.add(h); keep.append(r)
print(len(rows),"->",len(keep))
for r in keep: print(json.dumps({"messages":[{"role":"user","content":r["instruction"]},{"role":"assistant","content":r["output"]}]}))

4 -> 2
{"messages": [{"role": "user", "content": "Summarise: Server crashed at 2am."}, {"role": "assistant", "content": "The server crashed at 2am."}]}
{"messages": [{"role": "user", "content": "Summarise: Disk is 90% full."}, {"role": "assistant", "content": "The disk is nearly full."}]}


In [ ]:
# NOT RUN HERE: shape of a QLoRA run with Hugging Face libraries
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained("<base-model-id>", quantization_config=bnb)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj"], task_type="CAUSAL_LM")
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds,
                     peft_config=lora, args=SFTConfig(output_dir="out", num_train_epochs=2,
                     learning_rate=2e-4, per_device_train_batch_size=4))
trainer.train()

## 10.5 Evaluating, Merging and Serving Fine-Tuned Models

## 10.6 Alignment Maths: Reward Models, DPO, GRPO, Distillation and Quantisation

In [ ]:
import math
sig = lambda x: 1 / (1 + math.exp(-x))
for gap in (1.5, -1.5, 0.0):
    print(f"gap {gap:+.1f}  P(winner) = {sig(gap):.4f}  loss = {-math.log(sig(gap)):.4f}")

gap +1.5  P(winner) = 0.8176  loss = 0.2014
gap -1.5  P(winner) = 0.1824  loss = 1.7014
gap +0.0  P(winner) = 0.5000  loss = 0.6931


In [ ]:
import math
sig = lambda x: 1 / (1 + math.exp(-x))
def dpo(pw, rw, pl, rl, beta=0.1):
    m = beta * ((pw - rw) - (pl - rl))
    return m, -math.log(sig(m)), sig(-m)
print("pair above:   margin %.3f loss %.4f weight %.4f" % dpo(-12.0, -13.0, -15.0, -14.0))
print("at start:     margin %.3f loss %.4f weight %.4f" % dpo(-13.0, -13.0, -14.0, -14.0))
print("going wrong:  margin %.3f loss %.4f weight %.4f" % dpo(-14.0, -13.0, -13.0, -14.0))
print("beta=0.5:     margin %.3f loss %.4f weight %.4f" % dpo(-12.0, -13.0, -15.0, -14.0, beta=0.5))

pair above:   margin 0.200 loss 0.5981 weight 0.4502
at start:     margin 0.000 loss 0.6931 weight 0.5000
going wrong:  margin -0.200 loss 0.7981 weight 0.5498
beta=0.5:     margin 1.000 loss 0.3133 weight 0.2689


In [ ]:
import numpy as np
def advantages(r, eps=1e-8):
    r = np.array(r, float)
    return (r - r.mean()) / (r.std() + eps)
print(np.round(advantages([1, 0, 0, 1]), 3))
print(np.round(advantages([1, 0, 0, 0]), 3))
print(np.round(advantages([1, 1, 1, 1]), 3))

[ 1. -1. -1.  1.]
[ 1.732 -0.577 -0.577 -0.577]
[0. 0. 0. 0.]


In [ ]:
params = 70e9
print("fp16 GB:", params * 2 / 1e9)
print("int8 GB:", params * 1 / 1e9)
q4 = params * 0.5 + params / 64 * 2
print("4-bit g64 GB:", round(q4 / 1e9, 2), " bits/weight:", round(q4 * 8 / params, 3))

fp16 GB: 140.0
int8 GB: 70.0
4-bit g64 GB: 37.19  bits/weight: 4.25


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
W = rng.normal(size=(64, 256)) * 0.02
W[:, 7] *= 12                                  # one outlier channel
def quant(W, group):
    out = np.empty_like(W)
    for i in range(W.shape[0]):
        for j in range(0, W.shape[1], group):
            blk = W[i, j:j + group]
            s = np.abs(blk).max() / 7              # 4-bit signed: -7..7
            out[i, j:j + group] = np.round(blk / s) * s
    return out
for g in (256, 64, 16):
    err = np.sqrt(np.mean((W - quant(W, g)) ** 2))
    print(f"group size {g:3d}: RMS error {err:.5f}")
print("weight RMS:", round(float(np.sqrt(np.mean(W ** 2))), 5))

group size 256: RMS error 0.00804
group size  64: RMS error 0.00433
group size  16: RMS error 0.00249
weight RMS: 0.02359


## Chapter recap

## Practice set

Problem 1 (easy). True or false: fine-tuning is the best way to keep a bot up to date with this week's price list.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch10.html